# Introduction

This notebook presents the data aggregation phase of the analytical workflow. Its purpose is to transform the synthetic evaluation data generated previously into project-level analytical features through predefined aggregation procedures.

## 1-Data Loading and Overview

In [1]:
import pandas as pd

# Do not display warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# Simulation Dataset import and structural overview of all sheets
file_path = "/kaggle/input/datasets/jfjutras07/data-aggregation-dataset/2-Data_Aggregation_Dataset.xlsx"

xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

dataframes = {}

for sheet in sheet_names:
    dataframes[sheet] = pd.read_excel(xls, sheet_name=sheet)

overview = pd.DataFrame({
    "table_name": sheet_names,
    "n_rows": [dataframes[s].shape[0] for s in sheet_names],
    "n_cols": [dataframes[s].shape[1] for s in sheet_names]
})

overview

,table_name,n_rows,n_cols
0,Activity,196,7
1,Activity_Certainty_Score,196,7
2,Partner,5,2
3,Project_Certainty_Score,50,14
4,Project_Partner,390,14
5,Project_Volunteer,78,7


## 2-Aggregation of Activity-Level Metrics into Project-Level Features

This step aggregates activity-level evaluation scores into project-level features. Multiple statistical descriptors (mean, standard deviation, minimum, maximum, and median) are calculated to summarize both the central tendency and variability of project activities while preserving information relevant for subsequent analyses.

In [2]:
#Dataframe Naming
df_activity = dataframes["Activity"]
df_activity_cs = dataframes["Activity_Certainty_Score"]
df_partner = dataframes["Partner"]
df_project_cs = dataframes["Project_Certainty_Score"]
df_project_partner = dataframes["Project_Partner"]
df_project_volunteer = dataframes["Project_Volunteer"]

# Aggregating activity-level variables into project-level statistical descriptors
activity_agg = df_activity.groupby("id_project").agg({
    "cost": ["mean", "std", "min", "max", "median"],
    "time": ["mean", "std", "min", "max", "median"],
    "novelty": ["mean", "std", "min", "max", "median"],
    "effort": ["mean", "std", "min", "max", "median"],
})

# Flattening column structure for ML compatibility
activity_agg.columns = ["_".join(col) for col in activity_agg.columns]
activity_agg = activity_agg.reset_index()

# Replace undefined standard deviations (single observation) with zero
std_columns = [col for col in activity_agg.columns if col.endswith("_std")]
activity_agg[std_columns] = activity_agg[std_columns].fillna(0)

# Adding project size feature (number of activities per project)
activity_agg["n_activities"] = df_activity.groupby("id_project").size().values

# Quick validation of output structure
activity_agg.head()

,id_project,cost_mean,cost_std,cost_min,cost_max,cost_median,time_mean,time_std,time_min,time_max,...,novelty_std,novelty_min,novelty_max,novelty_median,effort_mean,effort_std,effort_min,effort_max,effort_median,n_activities
0,1,2.00,0.707107,1,3,2.0,2.400000,1.140175,1,4,...,1.224745,1,4,2.0,2.800000,1.643168,1,4,4.0,5
1,2,2.00,1.154701,1,3,2.0,2.000000,0.816497,1,3,...,1.500000,2,5,2.0,3.000000,1.154701,2,4,3.0,4
2,3,2.00,1.000000,1,3,2.0,1.666667,1.154701,1,3,...,0.577350,1,2,2.0,2.333333,0.577350,2,3,2.0,3
3,4,2.25,1.258306,1,4,2.0,1.750000,0.957427,1,3,...,0.957427,1,3,1.5,2.250000,1.258306,1,4,2.0,4
4,5,2.00,1.414214,1,3,2.0,3.000000,1.414214,2,4,...,0.000000,3,3,3.0,3.000000,1.414214,2,4,3.0,2


## 3-Activity-Level Credibility Score Aggregation

At the activity level, credibility scores (CS) represent the analyst's confidence in the quality of the evidence supporting each evaluation dimension (Cost, Time, Novelty, and Effort). The same aggregation procedure is applied to activity-level credibility scores to produce project-level descriptors that remain consistent with the activity features.

In [3]:
# Aggregating activity-level credibility scores into project-level statistical descriptors
activity_cs_agg = df_activity_cs.groupby("id_project").agg({
    "CS_cost": ["mean", "std", "min", "max", "median"],
    "CS_time": ["mean", "std", "min", "max", "median"],
    "CS_novelty": ["mean", "std", "min", "max", "median"],
    "CS_effort": ["mean", "std", "min", "max", "median"],
})

# Flattening column structure for ML compatibility
activity_cs_agg.columns = ["_".join(col) for col in activity_cs_agg.columns]
activity_cs_agg = activity_cs_agg.reset_index()

# Replace undefined standard deviations (single observation) with zero
std_columns = [col for col in activity_cs_agg.columns if col.endswith("_std")]
activity_cs_agg[std_columns] = activity_cs_agg[std_columns].fillna(0)

# Adding project size feature (number of activities per project)
activity_cs_agg["n_activities"] = df_activity_cs.groupby("id_project").size().values

# Quick validation of output structure
activity_cs_agg.head()

,id_project,CS_cost_mean,CS_cost_std,CS_cost_min,CS_cost_max,CS_cost_median,CS_time_mean,CS_time_std,CS_time_min,CS_time_max,...,CS_novelty_std,CS_novelty_min,CS_novelty_max,CS_novelty_median,CS_effort_mean,CS_effort_std,CS_effort_min,CS_effort_max,CS_effort_median,n_activities
0,1,2.80,1.095445,1,4,3.0,1.600000,1.341641,1,4,...,0.836660,3,5,4.0,1.400000,0.894427,1,3,1.0,5
1,2,1.75,0.500000,1,2,2.0,1.750000,0.957427,1,3,...,1.707825,1,5,3.5,1.250000,0.500000,1,2,1.0,4
2,3,2.00,1.732051,1,4,1.0,1.666667,1.154701,1,3,...,1.000000,1,3,2.0,2.666667,1.527525,1,4,3.0,3
3,4,3.25,0.500000,3,4,3.0,2.250000,1.258306,1,4,...,0.816497,2,4,3.0,2.250000,0.957427,1,3,2.5,4
4,5,3.00,1.414214,2,4,3.0,1.000000,0.000000,1,1,...,1.414214,2,4,3.0,1.000000,0.000000,1,1,1.0,2


## 4-Aggregation of Partner-Level Project Evaluations

Project-level evaluation data originate from two complementary assessment procedures. Collaboration and Stability are evaluated by volunteers, whereas the remaining project dimensions are assessed through partner group interviews. Scores are first aggregated at the partner level when multiple respondents are available and subsequently combined to obtain project-level features.

In [4]:
# Aggregating volunteer evaluations at partner level for multipartner projects only
# This ensures collaboration and stability are first computed per partner when multiple partners exist
volunteer_partner_scores = (
    df_project_volunteer
    .groupby(["id_project", "id_partner"])
    .agg({
        "collaboration": ["mean", "std", "min", "max", "median"],
        "stability": ["mean", "std", "min", "max", "median"]
    })
)

# Flatten column names for ML compatibility
volunteer_partner_scores.columns = [
    "_".join(col) for col in volunteer_partner_scores.columns
]
volunteer_partner_scores = volunteer_partner_scores.reset_index()

# Aggregating partner-level scores to project level (partner-based dimensions)
partner_project_scores = (
    df_project_partner
    .groupby("id_project")
    .agg({
        "use": ["mean", "std", "min", "max", "median"],
        "quality": ["mean", "std", "min", "max", "median"],
        "reach": ["mean", "std", "min", "max", "median"],
        "results": ["mean", "std", "min", "max", "median"],
        "satisfaction": ["mean", "std", "min", "max", "median"],
        "alignment": ["mean", "std", "min", "max", "median"],
        "durability": ["mean", "std", "min", "max", "median"],
        "resilience": ["mean", "std", "min", "max", "median"]
    })
)

# Flatten column names
partner_project_scores.columns = [
    "_".join(col) for col in partner_project_scores.columns
]
partner_project_scores = partner_project_scores.reset_index()

# Replace undefined standard deviations (single observation) with zero
std_columns = [col for col in partner_project_scores.columns if col.endswith("_std")]
partner_project_scores[std_columns] = partner_project_scores[std_columns].fillna(0)

# Aggregating volunteer-derived partner scores up to project level
volunteer_project_scores = (
    volunteer_partner_scores
    .groupby("id_project")
    .agg({
        "collaboration_mean": ["mean", "std", "min", "max", "median"],
        "stability_mean": ["mean", "std", "min", "max", "median"]
    })
)

# Flatten column names
volunteer_project_scores.columns = [
    "_".join(col) for col in volunteer_project_scores.columns
]
volunteer_project_scores = volunteer_project_scores.reset_index()

# Replace undefined standard deviations (single observation) with zero
std_columns = [col for col in volunteer_project_scores.columns if col.endswith("_std")]
volunteer_project_scores[std_columns] = volunteer_project_scores[std_columns].fillna(0)

# Final merge at project level
project_scores = partner_project_scores.merge(
    volunteer_project_scores,
    on="id_project",
    how="left"
)

# Quick validation
project_scores.head()

,id_project,use_mean,use_std,use_min,use_max,use_median,quality_mean,quality_std,quality_min,quality_max,...,collaboration_mean_mean,collaboration_mean_std,collaboration_mean_min,collaboration_mean_max,collaboration_mean_median,stability_mean_mean,stability_mean_std,stability_mean_min,stability_mean_max,stability_mean_median
0,1,2.4,0.547723,2,3,2.0,2.0,0.707107,1,3,...,4.0,0.0,4.0,4.0,4.0,5.0,0.0,5.0,5.0,5.0
1,2,1.6,0.547723,1,2,2.0,3.6,0.547723,3,4,...,5.0,0.0,5.0,5.0,5.0,5.0,0.0,5.0,5.0,5.0
2,3,2.6,0.894427,2,4,2.0,3.0,1.414214,2,5,...,3.0,0.0,3.0,3.0,3.0,4.0,0.0,4.0,4.0,4.0
3,4,2.6,0.894427,2,4,2.0,3.0,1.000000,2,4,...,2.0,0.0,2.0,2.0,2.0,5.0,0.0,5.0,5.0,5.0
4,5,3.8,0.836660,3,5,4.0,2.8,0.836660,2,4,...,1.0,0.0,1.0,1.0,1.0,4.0,0.0,4.0,4.0,4.0


## 5-Project-Level Credibility Score Feature Construction

This step derives two project-level credibility indicators from the aggregated activity-level credibility scores. Performance credibility combines Cost and Time credibility, whereas Complexity credibility combines Novelty and Effort credibility. These indicators provide a simplified representation of evidence reliability for subsequent analyses.

In [5]:
# Aggregate activity-level credibility scores at project level
project_cs_features = (
    df_activity_cs
    .groupby("id_project")
    .agg({
        "CS_cost": "mean",
        "CS_time": "mean",
        "CS_novelty": "mean",
        "CS_effort": "mean"
    })
    .reset_index()
)

# Construct performance credibility (cost + time)
project_cs_features["CS_performance"] = (
    project_cs_features["CS_cost"] + project_cs_features["CS_time"]
) / 2

# Construct complexity credibility (novelty + effort) 
project_cs_features["CS_complexity"] = (
    project_cs_features["CS_novelty"] + project_cs_features["CS_effort"]
) / 2

# Keep only final constructs
project_cs_features = project_cs_features[
    ["id_project", "CS_performance", "CS_complexity"]
]

# Clean replacement
df_project_cs = df_project_cs.drop(
    columns=["CS_performance", "CS_complexity"],
    errors="ignore"
)

# Merge
df_project_cs = df_project_cs.merge(
    project_cs_features,
    on="id_project",
    how="left"
)

# Quick Validation
df_project_cs.head()

,id_project,Projects,project_type,project_class,CS_collaboration,CS_stability,CS_use,CS_quality,CS_reach,CS_results,CS_satisfaction,CS_alignment,CS_durability,CS_resilience,CS_performance,CS_complexity
0,1,Analysis of Gender Inequality in the Community,Analysis,External - Structural,4,5,1,3,5,4,4,3,4,4,2.200000,2.600000
1,2,Gender Baseline Assessment,Assessment,Internal - Structural,3,2,3,2,3,3,3,5,1,2,1.750000,2.250000
2,3,Nutrition and Health Capacity Building Program,Capacity Building,External - Behavioral,4,3,3,3,3,4,4,3,5,5,1.833333,2.333333
3,4,Organizational Risk Management Framework,Framework,Internal - Structural,5,5,3,3,4,2,3,4,4,4,2.750000,2.625000
4,5,Annual Operational Plan,Plan,Internal - Structural,3,5,3,3,4,4,3,3,4,5,2.000000,2.000000


## 6-Conclusion

This aggregation step transformed the simulated evaluation data into structured project-level analytical features. Activity evaluations, credibility scores, and partner and volunteer assessments were consistently aggregated while preserving both central tendency and variability. The resulting datasets provide a coherent foundation for the subsequent integration of contextual variables and the application of machine learning models.

## 7-Data Export

In [6]:
import os

# Define export path
export_path = "/kaggle/working/"
excel_file = "simulation_aggregated_dataset.xlsx"
excel_path = os.path.join(export_path, excel_file)

# Ensure directory exists
os.makedirs(export_path, exist_ok=True)

# Define aggregated tables to export
tables_to_export = {
    "Activity_Agg": activity_agg,
    "Activity_CS_Agg": activity_cs_agg,
    "Project_Scores": project_scores,
    "Project_CS": df_project_cs
}

# Export all aggregated tables into a single Excel file
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df in tables_to_export.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Validation
print(f"Aggregated dataset successfully created: {excel_path}")

Aggregated dataset successfully created: /kaggle/working/simulation_aggregated_dataset.xlsx
